# NASA Exoplanet Archive Cross-match

**Purpose**: Match candidates against the NASA Confirmed Planets table and the TOI (TESS Objects of Interest) table, labeling them as 「Known / New Candidate」. This is **annotation for deduplication**, not physical filtering — it aids deduplication and prioritization, but does not affect the pass/fail decision.

> Note: 「Known」means the host/signal is already registered in the archive (either as a confirmed planet or a registered TOI); it does *not* mean this particular candidate signal has been conclusively confirmed as a planet.

**Matching strategy (name + TIC ID dual-key)**:
```
For each candidate:
  1. Query NASA Confirmed Planets table   query_object(target)     hit -> Known
  2. Miss -> query TOI table              by TIC_ID: where tid=... hit -> Known
  3. Both miss                            -> New Candidate
```
Name matching is intuitive but has alias risks; TIC ID is a numeric primary key, avoiding name ambiguity. TIC ID is reused directly from the radius check output.

**Input**: Prefers `radius_result.csv` (contains TIC_ID), falls back to `odd_even_result.csv` / Person1's original table.
**Output**: New column `Known/New_Candidate`, saved as `nasa_crossmatch_result.csv`. If all three validation columns are present, a high-confidence candidate list is also generated.

## 0. Install Dependencies

In [ ]:
%pip install -q numpy pandas astropy astroquery

## 1. Configuration and Data Loading

Preferentially read the output from the previous step (radius check), so that all three validation results accumulate in the same table.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from collections import Counter

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

# Input priority: radius check output -> odd-even output -> Person1 original table
_PATHS = ['radius_result.csv',
          'odd_even_result.csv',
          'master_candidates.csv',
          'attachments/master_candidates.csv']
INPUT_CSV = next((p for p in _PATHS if os.path.exists(p)), _PATHS[-1])
OUTPUT_CSV = 'nasa_crossmatch_result.csv'

PERSON1_COLS = ['Target', 'Sector', 'Period (d)', 'Period Err (d)',
                'Depth (ppm)', 'Duration (hr)', 'SNR', 'False Positive']

df = pd.read_csv(INPUT_CSV)
df_in = df.copy()
has_tic = 'TIC_ID' in df.columns
print(f'Loaded {INPUT_CSV} -> {len(df)} candidates   (has TIC_ID: {has_tic})')
print('Existing columns:', list(df.columns))
df.head()

## 2. Core Functions

`crossmatch_archive` first queries the Confirmed Planets table (by name), and if that misses, queries the TOI table (by TIC ID). Includes caching and fault tolerance, returning both the label and the hit source.

In [ ]:
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

_ARCHIVE_CACHE = {}

def _valid_tic(tic_id):
    try:
        return tic_id is not None and not pd.isna(tic_id) and int(tic_id) > 0
    except Exception:
        return False

def crossmatch_archive(target, tic_id=None):
    '''Return {label: Known/New_Candidate, source: confirmed/TOI/none/query_error}.'''
    key = (str(target), int(tic_id) if _valid_tic(tic_id) else None)
    if key in _ARCHIVE_CACHE:
        return _ARCHIVE_CACHE[key]
    label, source = 'New Candidate', 'none'
    # (1) NASA Confirmed Planets table: match by name (built-in alias resolution)
    try:
        t = NasaExoplanetArchive.query_object(str(target))
        if t is not None and len(t) > 0:
            label, source = 'Known', 'confirmed'
    except Exception:
        source = 'query_error'
    # (2) If missed, query TOI table: match by TIC ID (numeric primary key)
    if label == 'New Candidate' and _valid_tic(tic_id):
        try:
            q = NasaExoplanetArchive.query_criteria(
                table='toi', select='toi,tid,tfopwg_disp',
                where=f'tid={int(tic_id)}')
            if q is not None and len(q) > 0:
                label, source = 'Known', 'TOI'
        except Exception:
            if source != 'confirmed':
                source = 'query_error'
    out = {'label': label, 'source': source}
    _ARCHIVE_CACHE[key] = out
    return out

## 2.1 Single-Target Diagnostic (Run This Cell First)

Use the first candidate to verify connectivity to the NASA Exoplanet Archive. If it reports `query_error`, the issue is usually network/astroquery access.

In [ ]:
_t = str(df['Target'].iloc[0])
_tic = df['TIC_ID'].iloc[0] if has_tic else None
_r = crossmatch_archive(_t, _tic)
print(f'Target={_t}  TIC_ID={_tic}  ->  {_r}')
if _r['source'] == 'query_error':
    print('>> Hint: NASA Archive query errored; check network / astroquery access.')
else:
    print('>> Pipeline working; proceed to batch run.')

## 3. Batch Cross-match

In [ ]:
labels, sources = [], []
for i, row in df.iterrows():
    tic = row['TIC_ID'] if has_tic else None
    r = crossmatch_archive(row['Target'], tic)
    labels.append(r['label']); sources.append(r['source'])
    print(f"[{i+1:>2}/{len(df)}] {str(row['Target']):<12} -> {r['label']:<6} (source: {r['source']})")

df['Known/New_Candidate'] = labels
df['match_source'] = sources

print('\nCross-match completed ->  Known:', labels.count('Known'), '  New Candidate:', labels.count('New Candidate'))
print('Hit source distribution:')
for s, cnt in Counter(sources).most_common():
    print(f'   {cnt:>3}  x  {s}')

## 4. Save Results and Final High-Confidence List

If all three validation columns (odd-even, radius, cross-match) are present, output the final high-confidence list: **odd-even pass AND radius plausible AND not Person1 original false positive**.

In [ ]:
# Integrity check: Person1's original columns must remain unchanged
_p1 = [c for c in PERSON1_COLS if c in df.columns and c in df_in.columns]
assert df[_p1].equals(df_in[_p1]), 'Person1 original data was unexpectedly modified!'
print('Original data integrity check passed.\n')

df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print('Saved complete annotated table ->', OUTPUT_CSV)

# If all three validations are present, generate the final high-confidence list
if {'odd_even_pass', 'planet_radius_plausible'}.issubset(df.columns):
    fp = df['False Positive'].astype(str).str.lower().isin(['true', '1'])
    mask = (df['odd_even_pass'] == True) & (df['planet_radius_plausible'] == True) & (~fp)
    high_conf = df[mask.values].copy()
    high_conf.to_csv('high_confidence_candidates.csv', index=False, encoding='utf-8-sig')
    print('\n' + '=' * 60)
    print('Three-Check Summary')
    print('=' * 60)
    print(f'Total candidates        : {len(df)}')
    print(f'Odd-even pass            : {int((df["odd_even_pass"]==True).sum())}')
    print(f'Radius plausible         : {int((df["planet_radius_plausible"]==True).sum())}')
    print(f'Known / New Candidate     : {labels.count("Known")} / {labels.count("New Candidate")}')
    print(f'>> High-confidence count  : {len(high_conf)}  -> high_confidence_candidates.csv')
    print('=' * 60)
else:
    print('\nHint: The current input does not contain odd-even/radius check columns (possibly using Person1 original table directly),')
    print('      only cross-match annotation completed. For the high-confidence list, please run the first two notebooks first.')
    high_conf = None
df

## 5. Notes

- `Known/New_Candidate`: A hit in the NASA Confirmed Planets table or the TOI table = 「Known」; otherwise 「New Candidate」. `match_source` records the hit source (confirmed/TOI/none/query_error).
- This project's targets were deliberately chosen from known TESS planet hosts, so a large number of 「Known」is **expected**; if 「New Candidate」appears, first suspect a name-alias matching failure and verify with coordinates or TIC ID.
- This column is annotation-only (deduplication/priority) and does not participate in the pass/fail filter. The final high-confidence list is determined solely by the odd-even check + radius check + the original false positive flag.
- Output `nasa_crossmatch_result.csv` has accumulated all three validation results, i.e., the complete annotated data table.